# Example: CSerpentModule

`CSerpentModule` lets you develop a C extension inside a Jupyter notebook:
write C in a cell, compile it, and reload it on the fly as you edit.

The interface is exactly the one the standalone `cserpent` program uses. You
mark up your C with `CSERPENT_*` annotations inside `#ifdef CSERPENT` blocks,
which vanish from every ordinary build. That is deliberate: notebook code is
usually a draft of something that ends up in a real `.c` file, and porting it
should be copy and paste with nothing to rewrite.

## Installing the C-serpent extension module

C-serpent is primarily a standalone command-line program, but it can also be
built as a Python extension module so that it can be called directly from
Python. `cserpentmodule` needs that extension.

Install it with `pip install cserpent`, or build it yourself. The `pip` package
is a *source package*, so you need a C compiler and Python development headers
available for it to install.

To build it by hand, in a UNIX-like shell:

    PY_INC_DIR="$(python -c 'import sysconfig; print(sysconfig.get_paths()["include"])')"

    cc -shared -fPIC -o cserpent_py.so cserpent_py.c -I$PY_INC_DIR

The first line asks Python where its headers are. You may need to install them;
on Linux they are usually in a package called `python3-dev` or `python3-devel`.
The second line builds the extension module.

## Compile and wrap C code

In [ ]:
import cserpentmodule, numpy

In [ ]:
c_code = """

#include <stdint.h>

void add_array_scalar_i32(int N, int32_t *x, int32_t y) {
    for(int i = 0; i < N; i++)
        x[i] += y;
}

#ifdef CSERPENT
CSERPENT_WRAPFN(add_array_scalar_i32)
#endif

"""

The `#ifdef CSERPENT` block is the only thing that tells C-serpent what to
wrap. `CSERPENT_MODULE` is *not* written here: the module name has to change on
every build for reloading to work, so `CSerpentModule` supplies it. That one
line is all you add when you move this code into a `.c` file.

In [ ]:
m = cserpentmodule.CSerpentModule("my_c_module")
my_c_module = m.compile(c_code)

In [ ]:
x = numpy.zeros(5, numpy.int32)
my_c_module.add_array_scalar_i32(len(x), x, 3)
x

## Update C code and live-reload

Just call `compile` again. Each build gets a fresh module name behind the
scenes, so you can reload as often as you like with no waiting.

In [ ]:
updated_c_code = """

#include <stdint.h>

void add_array_scalar_i32(int N, int32_t *x, int32_t y) {
    for(int i = 0; i < N; i++)
        x[i] += y;
}

void mul_array_scalar_i32(int N, int32_t *x, int32_t y) {
    for(int i = 0; i < N; i++)
        x[i] *= y;
}

#ifdef CSERPENT
CSERPENT_WRAPFN(add_array_scalar_i32, mul_array_scalar_i32)
#endif

"""

In [ ]:
my_c_module = m.compile(updated_c_code)

In [ ]:
[n for n in dir(my_c_module) if not n.startswith('_')]

In [ ]:
x = numpy.ones(5, numpy.int32)
my_c_module.mul_array_scalar_i32(len(x), x, 3)
x

## Structs, constants, and generic functions

`CSERPENT_WRAPTYPE` turns a struct into a Python class. Scalar members are
readable and writable; nested structs and fixed-size arrays come back as views
that write through to the parent.

`CSERPENT_WRAPCONST` exports enum constants, and `CSERPENT_WRAPFN_GENERIC`
builds one dispatcher over a family of type-suffixed functions.

In [ ]:
richer = """

#include <stdint.h>

enum status { ST_IDLE, ST_BUSY };

typedef struct {
    int32_t rows;
    int32_t cols;
    double  scale;
    double  coeffs[4];
} Matrix;

double matrix_trace(Matrix *m) { return m->rows * m->scale; }

double meanf(int N, float  *a) { double s=0; for(int i=0;i<N;i++) s+=a[i]; return s/N; }
double meand(int N, double *a) { double s=0; for(int i=0;i<N;i++) s+=a[i]; return s/N; }

#ifdef CSERPENT
CSERPENT_WRAPTYPE(Matrix, readonly = (rows, cols))
CSERPENT_WRAPFN(matrix_trace)
CSERPENT_WRAPFN_GENERIC(mean)
CSERPENT_WRAPCONST(status)
#endif

"""

mats = cserpentmodule.CSerpentModule("mats").compile(richer)

In [ ]:
mat = mats.Matrix(scale=2.0)
mat.coeffs[:] = [1, 2, 3, 4]     # array member is a view, so this writes through
print(mat)
print("coeffs      ", mat.coeffs)
print("constants   ", mats.ST_IDLE, mats.ST_BUSY)
print("generic f32 ", mats.mean(3, numpy.array([1,2,3], dtype=numpy.float32)))
print("generic f64 ", mats.mean(3, numpy.array([1,2,3], dtype=numpy.float64)))

In [ ]:
# rows and cols were declared readonly, so C can set them but Python cannot
try:
    mat.rows = 5
except AttributeError as e:
    print(e)

## A shortcut: the `%%cserpent` cell magic

Importing `cserpentmodule` registers a cell magic, so you can skip the Python
string entirely. The cell body is plain C, which means you can select it and
paste it straight into a `.c` file.

In [ ]:
%%cserpent quick

#include <stdint.h>

int64_t add_i64(int64_t a, int64_t b) { return a + b; }

#ifdef CSERPENT
CSERPENT_WRAPFN(add_i64, doc = "Add two 64-bit integers.")
#endif

In [ ]:
print(quick.add_i64(3, 4))
print(quick.add_i64.__doc__)

## Moving this into a real project

Copy the C out of the cell into a `.c` file and add a module name:

```c
#ifdef CSERPENT
CSERPENT_MODULE(my_c_module)
CSERPENT_WRAPFN(add_array_scalar_i32)
#endif
```

then build it with the standalone tool:

```
$ cserpent my_c_file.c > wrappers.c
$ cc -fPIC -shared -I$(python -c 'import sysconfig; print(sysconfig.get_paths()["include"])') \
     -I$(python -c 'import numpy; print(numpy.get_include())') \
     wrappers.c my_c_file.c -o my_c_module.so
```

Nothing else changes. See the README for the full set of annotations.